# Supplementary Figure — joint decoding and clustering using the new held-out top-\(m\) results

This notebook updates the old clustering analysis so that the **archetype subsets
come from the revised participant-level cross-fitted top-\(m\) analysis** rather
than from the older same-data top-\(m\) ranking.

## What changes

For each model order \(K\), ranking condition, and subset size \(m\):

1. Read the selected archetypes from `heldout_topm_crossfit_runs.csv`.
2. Combine the repeated A→B / B→A cross-fitted selections into a
   **consensus held-out-selected subset** using selection frequency.
3. Use the held-out decoding accuracy from the revised top-\(m\) analysis.
4. Re-run the original condition-structure validation using the consensus subset:
   - spatial AA features = `sXC[:, selected_archetypes]`
   - each participant is one observation
   - compare the selected subset with random subsets of identical size from the
     same model order \(K\)
5. Recompute the original joint score:
   \[
   z(\mathrm{held\mbox{-}out\ decoding}) +
   z(\mathrm{clustering\ advantage})
   \]

The resulting two-panel figure preserves the logic of the old clustering figure,
but the subset-selection step is now based on the revised cross-fitted top-\(m\)
results.

### Important interpretation point

This is **cross-fitted subset selection**, not a fully held-out clustering test.
The selected archetypes are derived from repeated participant-level ranking
splits, but the clustering metric is then evaluated across the complete
condition-subject set, as in the original clustering validation. That makes this
a clean replacement for the old figure without changing the clustering target.

## Safe to interrupt and resume

The expensive clustering/null loop in this version is **checkpointed**.

Each `(condition, K, top-m)` combination is written to its own file under:

`outputs/paper_supplement/clustering_checkpoints/`

During an unfinished job, progress is stored in a `.partial.csv` every `CHECKPOINT_EVERY` random subsets. When a job reaches all requested random subsets, it is promoted to a completed `.csv`.

If the kernel, browser, computer, or notebook is interrupted, rerun the setup/loading cells and then rerun the expensive clustering cell. Completed jobs are skipped automatically, and a partial job resumes from the saved random-subset index.

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

%matplotlib inline

from pathlib import Path
import ast
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_score,
)
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

In [ ]:
# ============================================================
# LOCATE THE NEW HELD-OUT TOP-M OUTPUTS
# ============================================================

RUN_NAME = "heldout_topm_crossfit_runs.csv"
RANK_NAME = "heldout_per_archetype_ranking_runs.csv"

candidate_run = [
    Path(RUN_NAME),
    Path("msaa_heldout_topm_outputs_spatial_across") / RUN_NAME,
    Path("../msaa_heldout_topm_outputs_spatial_across") / RUN_NAME,
    Path("../../msaa_heldout_topm_outputs_spatial_across") / RUN_NAME,
]

candidate_rank = [
    Path(RANK_NAME),
    Path("msaa_heldout_topm_outputs_spatial_across") / RANK_NAME,
    Path("../msaa_heldout_topm_outputs_spatial_across") / RANK_NAME,
    Path("../../msaa_heldout_topm_outputs_spatial_across") / RANK_NAME,
]


def first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None


run_path = first_existing(candidate_run)
rank_path = first_existing(candidate_rank)

# Bounded recursive fallback.
if run_path is None:
    for root in [Path.cwd(), Path.cwd().parent]:
        try:
            hits = list(root.glob(f"**/{RUN_NAME}"))
        except (OSError, PermissionError):
            hits = []
        if hits:
            run_path = hits[0]
            break

if rank_path is None:
    for root in [Path.cwd(), Path.cwd().parent]:
        try:
            hits = list(root.glob(f"**/{RANK_NAME}"))
        except (OSError, PermissionError):
            hits = []
        if hits:
            rank_path = hits[0]
            break

if run_path is None:
    raise FileNotFoundError(
        f"Could not find {RUN_NAME}. Run the revised held-out top-m notebook first."
    )

run_df = pd.read_csv(run_path)

rank_df = (
    pd.read_csv(rank_path)
    if rank_path is not None
    else pd.DataFrame()
)

print("Held-out run table:", run_path)
print("Ranking table:", rank_path)
print("Available K:", sorted(run_df["K"].unique()))
print("Available top-m:", sorted(run_df["top_m"].unique()))

In [ ]:
# ============================================================
# SETTINGS
# ============================================================

FIT_SCOPE = "across"
ANALYSIS_TYPE = "spatial"
CONDITIONS = ["intact", "word", "rest"]

# Paper-focused subset sizes.
# Change to [1, 2, 3, 5, 10] if you want the fuller sensitivity version.
TOP_M_VALUES = [3, 5, 10]

PRIMARY_CLUSTER_METRIC = "nmi"

N_RANDOM_SUBSETS = 500
RANDOM_SEED = 0

# ------------------------------------------------------------
# RESUMABLE CHECKPOINTING
# ------------------------------------------------------------
# Each (condition, K, top_m) job is saved separately.
# If execution is interrupted, rerunning the notebook will:
#   1. skip completed jobs
#   2. resume partially completed jobs from the last saved random subset
#
# At most CHECKPOINT_EVERY - 1 random subsets can be lost in an abrupt stop.
CHECKPOINT_EVERY = 20
FORCE_RECOMPUTE = False

CHECKPOINT_DIR = OUTPUT_DIR / "clustering_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

N_CLUSTERS = 3
CLUSTER_METHOD = "spectral"   # "spectral" or "kmeans"
N_INIT = 50

FEATURE_MODE = "flatten"      # matches the old clustering notebook
SCALE_FEATURES = True
USE_PCA = False
PCA_N_COMPONENTS = 20

SKIP_IF_TOP_M_GE_N_ARCHETYPES = True

SPATIAL_DENOMINATOR = 700

# Quick smoke-test mode.
# If True, uses only K=46 and 50 random subsets.
QUICK_TEST = False
QUICK_K_VALUES = [46]
QUICK_N_RANDOM_SUBSETS = 50

OUTPUT_DIR = Path("outputs/paper_supplement")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_CSV = OUTPUT_DIR / "new_topm_crossfit_clustering_summary.csv"
LONG_CSV = OUTPUT_DIR / "new_topm_crossfit_clustering_long.csv"
CONSENSUS_CSV = OUTPUT_DIR / "new_topm_crossfit_consensus_subsets.csv"

FIG_PNG = OUTPUT_DIR / "Supplement_joint_decoding_clustering_new_topm.png"
FIG_PDF = OUTPUT_DIR / "Supplement_joint_decoding_clustering_new_topm.pdf"

DPI = 300

In [ ]:
# ============================================================
# CLEAN THE HELD-OUT RUN TABLE
# ============================================================

def parse_selected_archetypes(x):
    if isinstance(x, list):
        return [int(v) for v in x]
    if isinstance(x, np.ndarray):
        return [int(v) for v in x.tolist()]
    if pd.isna(x):
        return []
    if isinstance(x, str):
        try:
            return [int(v) for v in ast.literal_eval(x)]
        except Exception:
            return [int(v) for v in re.findall(r"-?\d+", x)]
    return []


run = run_df.copy()

run["analysis_type"] = run["analysis_type"].astype(str)
run["fit_scope"] = run["fit_scope"].astype(str)
run["condition"] = run["condition"].astype(str)
run["K"] = run["K"].astype(int)
run["top_m"] = run["top_m"].astype(int)

run = run[
    (run["analysis_type"] == ANALYSIS_TYPE)
    & (run["fit_scope"] == FIT_SCOPE)
    & (run["condition"].isin(CONDITIONS))
].copy()

# Some small-K configurations can appear twice because requested top-m values
# collapse to the same actual m=min(requested_m, K). Remove exact duplicates
# before computing frequencies or uncertainty.
dedupe_cols = [
    "condition",
    "K",
    "top_m",
    "outer_rep",
    "direction",
    "outer_seed",
    "selected_archetypes",
    "mean",
]

run = (
    run
    .drop_duplicates(subset=[c for c in dedupe_cols if c in run.columns])
    .copy()
)

run["selected_archetype_list"] = (
    run["selected_archetypes"]
    .apply(parse_selected_archetypes)
)

available_topm = sorted(run["top_m"].unique())
TOP_M_VALUES = [
    m for m in TOP_M_VALUES
    if m in available_topm
]

if len(TOP_M_VALUES) == 0:
    raise ValueError(
        "None of the requested TOP_M_VALUES are available in the held-out table."
    )

if QUICK_TEST:
    run = run[
        run["K"].isin(QUICK_K_VALUES)
    ].copy()

print("Top-m values used:", TOP_M_VALUES)
print("K values used:", sorted(run["K"].unique()))
print("Rows after cleanup:", len(run))

In [ ]:
# ============================================================
# HELD-OUT DECODING SUMMARY
#
# A→B and B→A are first averaged within each outer split.
# SEM is then computed over outer repetitions.
# ============================================================

def sem(x):
    x = pd.Series(x).dropna()
    if len(x) < 2:
        return np.nan
    return x.std(ddof=1) / np.sqrt(len(x))


outer_decode = (
    run[
        run["top_m"].isin(TOP_M_VALUES)
    ]
    .groupby(
        ["condition", "K", "top_m", "outer_rep"],
        as_index=False,
    )
    .agg(
        heldout_accuracy=("mean", "mean"),
    )
)

decode_summary = (
    outer_decode
    .groupby(
        ["condition", "K", "top_m"],
        as_index=False,
    )
    .agg(
        decode_mean=("heldout_accuracy", "mean"),
        decode_err=("heldout_accuracy", sem),
        n_outer_reps=("outer_rep", "nunique"),
    )
)

display(decode_summary.head())

In [ ]:
# ============================================================
# CONSENSUS CROSS-FITTED SUBSETS
#
# Selection frequency is calculated over the repeated cross-fit ranking
# directions. Ties are broken using mean single-archetype ranking accuracy.
# ============================================================

ranking_tiebreak = pd.DataFrame()

if len(rank_df):
    r = rank_df.copy()

    r["analysis_type"] = r["analysis_type"].astype(str)
    r["fit_scope"] = r["fit_scope"].astype(str)
    r["condition"] = r["condition"].astype(str)
    r["K"] = r["K"].astype(int)
    r["archetype"] = r["archetype"].astype(int)

    r = r[
        (r["analysis_type"] == ANALYSIS_TYPE)
        & (r["fit_scope"] == FIT_SCOPE)
        & (r["condition"].isin(CONDITIONS))
    ].copy()

    ranking_tiebreak = (
        r
        .groupby(
            ["condition", "K", "archetype"],
            as_index=False,
        )
        .agg(
            rank_accuracy=("mean_accuracy", "mean"),
            rank_mean=("rank_mean", "mean"),
        )
    )


consensus_rows = []
frequency_rows = []

for (condition, K, top_m), sub in (
    run[
        run["top_m"].isin(TOP_M_VALUES)
    ]
    .groupby(
        ["condition", "K", "top_m"]
    )
):
    K = int(K)
    top_m = int(top_m)

    # Skip full-model subsets; they have no meaningful random same-size null.
    if SKIP_IF_TOP_M_GE_N_ARCHETYPES and top_m >= K:
        continue

    counts = np.zeros(K, dtype=float)

    for selected in sub["selected_archetype_list"]:
        for a in selected:
            if 0 <= int(a) < K:
                counts[int(a)] += 1

    n_selections = len(sub)

    freq = counts / max(n_selections, 1)

    freq_df = pd.DataFrame({
        "condition": condition,
        "K": K,
        "top_m": top_m,
        "archetype": np.arange(K, dtype=int),
        "selection_frequency": freq,
    })

    if len(ranking_tiebreak):
        tie = ranking_tiebreak[
            (ranking_tiebreak["condition"] == condition)
            & (ranking_tiebreak["K"] == K)
        ][
            [
                "archetype",
                "rank_accuracy",
                "rank_mean",
            ]
        ]

        freq_df = freq_df.merge(
            tie,
            on="archetype",
            how="left",
        )
    else:
        freq_df["rank_accuracy"] = np.nan
        freq_df["rank_mean"] = np.nan

    # Frequency is primary. Mean ranking accuracy is only a tie-break.
    freq_df = freq_df.sort_values(
        [
            "selection_frequency",
            "rank_accuracy",
            "rank_mean",
            "archetype",
        ],
        ascending=[
            False,
            False,
            False,
            True,
        ],
    )

    selected = (
        freq_df
        .head(top_m)["archetype"]
        .astype(int)
        .tolist()
    )

    selected_freq = (
        freq_df
        .head(top_m)["selection_frequency"]
        .to_numpy(dtype=float)
    )

    consensus_rows.append({
        "condition": condition,
        "K": K,
        "top_m": top_m,
        "selected_archetypes": str(selected),
        "selected_archetype_list": selected,
        "n_crossfit_selections": int(n_selections),
        "mean_selected_frequency": float(np.mean(selected_freq)),
        "min_selected_frequency": float(np.min(selected_freq)),
    })

    frequency_rows.append(freq_df)


consensus_df = pd.DataFrame(consensus_rows)

selection_frequency_df = (
    pd.concat(frequency_rows, ignore_index=True)
    if len(frequency_rows)
    else pd.DataFrame()
)

consensus_df = consensus_df.merge(
    decode_summary,
    on=[
        "condition",
        "K",
        "top_m",
    ],
    how="left",
)

consensus_df["component_ratio"] = (
    consensus_df["K"].astype(float)
    / SPATIAL_DENOMINATOR
)

consensus_df.to_csv(
    CONSENSUS_CSV,
    index=False,
)

print("Consensus rows:", len(consensus_df))
print("Saved:", CONSENSUS_CSV)

display(
    consensus_df
    .sort_values(
        ["condition", "K", "top_m"]
    )
    .head(20)
)

In [ ]:
# ============================================================
# LOAD THE ACROSS-CONDITION MS-AA FITS
# ============================================================

def find_msaa_npz(K):
    exact_names = [
        f"spatialAA_across_acrossCond_K{K}.npz",
    ]

    candidate_dirs = [
        Path("msaa_flexible_outputs_npz"),
        Path("."),
        Path("../msaa_flexible_outputs_npz"),
        Path("../../msaa_flexible_outputs_npz"),
    ]

    for d in candidate_dirs:
        for name in exact_names:
            p = d / name
            if p.exists():
                return p

    for root in [Path.cwd(), Path.cwd().parent]:
        try:
            hits = list(
                root.glob(
                    f"**/spatialAA_across_acrossCond_K{K}.npz"
                )
            )
        except (OSError, PermissionError):
            hits = []

        if hits:
            return hits[0]

    return None


def load_msaa_npz(path):
    data = np.load(path, allow_pickle=True)

    results_subj = data["results_subj"].tolist()

    if isinstance(results_subj, np.ndarray):
        results_subj = results_subj.tolist()

    labels = np.asarray(
        data["condition_labels_str"]
    ).astype(str)

    return {
        "results_subj": results_subj,
        "condition_labels_str": labels,
        "path": path,
    }


fits = {}

for K in sorted(consensus_df["K"].unique()):
    p = find_msaa_npz(int(K))

    if p is None:
        print(f"Missing fit for K={K}")
        continue

    fit = load_msaa_npz(p)

    if len(fit["results_subj"]) != len(
        fit["condition_labels_str"]
    ):
        raise ValueError(
            f"K={K}: results_subj and condition labels differ in length."
        )

    fits[int(K)] = fit

print("Loaded K values:", sorted(fits.keys()))

In [ ]:
# ============================================================
# CONDITION-SUBJECT FEATURE MATRIX
#
# Matches the old clustering notebook:
# spatial AA uses sXC[:, selected_archetypes].
# ============================================================

def to_float_array(x):
    return np.asarray(x, dtype=float)


def get_expression_matrix_for_observation(
    sub_result,
    selected_archetypes,
):
    sXC = to_float_array(
        sub_result["sXC"]
    )

    return sXC[
        :,
        [int(a) for a in selected_archetypes],
    ]


def summarize_expression_matrix(M):
    M = np.asarray(M, dtype=float)

    return np.concatenate([
        M.mean(axis=0),
        M.std(axis=0),
        M.min(axis=0),
        M.max(axis=0),
    ])


def build_observation_feature_matrix(
    results_subj,
    selected_archetypes,
):
    rows = []

    for sub in results_subj:
        M = get_expression_matrix_for_observation(
            sub,
            selected_archetypes,
        )

        M = np.nan_to_num(
            M,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )

        if FEATURE_MODE == "flatten":
            rows.append(
                M.ravel()
            )
        elif FEATURE_MODE == "summary":
            rows.append(
                summarize_expression_matrix(M)
            )
        else:
            raise ValueError(
                "FEATURE_MODE must be 'flatten' or 'summary'."
            )

    X = np.vstack(rows)

    if SCALE_FEATURES:
        X = StandardScaler().fit_transform(X)

    if USE_PCA and X.shape[1] > PCA_N_COMPONENTS:
        n_comp = min(
            PCA_N_COMPONENTS,
            X.shape[0] - 1,
            X.shape[1],
        )

        X = PCA(
            n_components=n_comp,
            random_state=RANDOM_SEED,
        ).fit_transform(X)

    return X

In [ ]:
# ============================================================
# CLUSTERING METRICS
# ============================================================

def cluster_samples(
    X,
    n_clusters=N_CLUSTERS,
    seed=RANDOM_SEED,
):
    X = np.asarray(
        X,
        dtype=float,
    )

    if CLUSTER_METHOD == "kmeans":
        return KMeans(
            n_clusters=n_clusters,
            random_state=seed,
            n_init=N_INIT,
        ).fit_predict(X)

    if CLUSTER_METHOD == "spectral":
        sim = np.corrcoef(X)

        sim = np.nan_to_num(
            sim,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )

        sim = np.clip(
            sim,
            -1.0,
            1.0,
        )

        aff = (
            sim + 1.0
        ) / 2.0

        np.fill_diagonal(
            aff,
            1.0,
        )

        return SpectralClustering(
            n_clusters=n_clusters,
            affinity="precomputed",
            assign_labels="kmeans",
            random_state=seed,
        ).fit_predict(aff)

    raise ValueError(
        "CLUSTER_METHOD must be 'kmeans' or 'spectral'."
    )


def cluster_purity(
    true_labels,
    pred_labels,
):
    true_labels = np.asarray(
        true_labels
    )

    pred_labels = np.asarray(
        pred_labels
    )

    true_vals = pd.unique(
        true_labels
    )

    pred_vals = pd.unique(
        pred_labels
    )

    table = np.zeros(
        (
            len(pred_vals),
            len(true_vals),
        ),
        dtype=int,
    )

    for i, p in enumerate(pred_vals):
        for j, t in enumerate(true_vals):
            table[i, j] = np.sum(
                (pred_labels == p)
                & (true_labels == t)
            )

    row_ind, col_ind = linear_sum_assignment(
        -table
    )

    return float(
        table[
            row_ind,
            col_ind,
        ].sum()
        / len(true_labels)
    )


def cluster_balance(
    pred_labels,
):
    _, counts = np.unique(
        pred_labels,
        return_counts=True,
    )

    if len(counts) <= 1:
        return 0.0

    expected = (
        len(pred_labels)
        / len(counts)
    )

    imbalance = (
        np.sum(
            np.abs(
                counts - expected
            )
        )
        / (
            2
            * len(pred_labels)
            * (
                1
                - 1 / len(counts)
            )
        )
    )

    return float(
        1 - imbalance
    )


def nearest_centroid_accuracy(
    X,
    labels,
):
    X = np.asarray(
        X,
        dtype=float,
    )

    labels = np.asarray(
        labels
    )

    unique = pd.unique(
        labels
    )

    correct = 0

    for i in range(len(X)):
        train = np.ones(
            len(X),
            dtype=bool,
        )

        train[i] = False

        cents = []
        labs = []

        for lab in unique:
            mask = (
                (labels == lab)
                & train
            )

            if np.sum(mask):
                cents.append(
                    X[mask].mean(
                        axis=0
                    )
                )

                labs.append(
                    lab
                )

        cents = np.vstack(
            cents
        )

        pred = labs[
            int(
                np.argmin(
                    cdist(
                        X[[i]],
                        cents,
                    ).ravel()
                )
            )
        ]

        correct += int(
            pred == labels[i]
        )

    return float(
        correct
        / len(labels)
    )


def evaluate_clustering(
    X,
    labels,
    seed=RANDOM_SEED,
):
    pred = cluster_samples(
        X,
        n_clusters=N_CLUSTERS,
        seed=seed,
    )

    purity = cluster_purity(
        labels,
        pred,
    )

    balance = cluster_balance(
        pred
    )

    out = {
        "purity": purity,
        "balance": balance,
        "purity_x_balance": float(
            purity * balance
        ),
        "ari": float(
            adjusted_rand_score(
                labels,
                pred,
            )
        ),
        "nmi": float(
            normalized_mutual_info_score(
                labels,
                pred,
            )
        ),
        "nearest_centroid_accuracy": (
            nearest_centroid_accuracy(
                X,
                labels,
            )
        ),
    }

    try:
        out["silhouette_true_labels"] = float(
            silhouette_score(
                X,
                labels,
            )
        )
    except Exception:
        out["silhouette_true_labels"] = np.nan

    try:
        out["silhouette_cluster_labels"] = float(
            silhouette_score(
                X,
                pred,
            )
        )
    except Exception:
        out["silhouette_cluster_labels"] = np.nan

    return out

In [ ]:
# ============================================================
# RUN CONSENSUS HELD-OUT-SELECTED SUBSETS VS RANDOM SUBSETS
# — RESUMABLE / CHECKPOINTED VERSION
# ============================================================

import os


def atomic_to_csv(df, path):
    """
    Write a CSV through a temporary file, then replace the destination.
    This reduces the chance of leaving a corrupted checkpoint if execution
    is interrupted during the write itself.
    """
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)


def job_stem(condition, K, top_m):
    return f"{condition}_K{int(K)}_top{int(top_m)}"


n_random = (
    QUICK_N_RANDOM_SUBSETS
    if QUICK_TEST
    else N_RANDOM_SUBSETS
)

skipped = []

jobs = (
    consensus_df[
        consensus_df["top_m"].isin(TOP_M_VALUES)
    ]
    .sort_values(["condition", "K", "top_m"])
    .reset_index(drop=True)
)

print(
    f"Checkpoint directory: {CHECKPOINT_DIR}\n"
    f"Jobs to consider: {len(jobs)}\n"
    f"Random subsets per job: {n_random}\n"
    f"Checkpoint every {CHECKPOINT_EVERY} random subsets"
)

for job_i, row in jobs.iterrows():

    K = int(row["K"])
    condition = str(row["condition"])
    top_m = int(row["top_m"])

    stem = job_stem(condition, K, top_m)
    complete_path = CHECKPOINT_DIR / f"{stem}.csv"
    partial_path = CHECKPOINT_DIR / f"{stem}.partial.csv"

    # --------------------------------------------------------
    # Completed job: skip it on rerun
    # --------------------------------------------------------
    if complete_path.exists() and not FORCE_RECOMPUTE:
        existing = pd.read_csv(complete_path)
        n_done = int(
            np.sum(
                existing["subset_type"].astype(str) == "random"
            )
        )
        print(
            f"[{job_i + 1}/{len(jobs)}] SKIP complete: "
            f"{condition} | K={K} | top-{top_m} | random={n_done}"
        )
        continue

    if FORCE_RECOMPUTE:
        for p in [complete_path, partial_path]:
            if p.exists():
                p.unlink()

    # --------------------------------------------------------
    # Validate fit + selected subset
    # --------------------------------------------------------
    selected = parse_selected_archetypes(
        row["selected_archetypes"]
    )

    if K not in fits:
        skipped.append({
            "condition": condition,
            "K": K,
            "top_m": top_m,
            "reason": "missing_fit",
        })
        print(
            f"[{job_i + 1}/{len(jobs)}] SKIP missing fit: "
            f"{condition} | K={K} | top-{top_m}"
        )
        continue

    fit = fits[K]
    results_subj = fit["results_subj"]
    labels = np.asarray(
        fit["condition_labels_str"]
    ).astype(str)

    n_arch = int(
        np.asarray(
            results_subj[0]["sXC"]
        ).shape[1]
    )

    selected = [
        int(a)
        for a in selected
        if 0 <= int(a) < n_arch
    ]

    actual_m = len(selected)

    if actual_m == 0:
        skipped.append({
            "condition": condition,
            "K": K,
            "top_m": top_m,
            "reason": "empty_selected",
        })
        continue

    if (
        SKIP_IF_TOP_M_GE_N_ARCHETYPES
        and actual_m >= n_arch
    ):
        skipped.append({
            "condition": condition,
            "K": K,
            "top_m": top_m,
            "actual_m": actual_m,
            "n_archetypes": n_arch,
            "reason": "top_m_ge_n_archetypes",
        })
        continue

    # --------------------------------------------------------
    # Shared metadata for this job
    # --------------------------------------------------------
    base = {
        "analysis_type": ANALYSIS_TYPE,
        "fit_scope": FIT_SCOPE,
        "K": K,
        "component_ratio": (
            K / SPATIAL_DENOMINATOR
        ),
        "condition": condition,
        "top_m": top_m,
        "actual_m": actual_m,
        "n_archetypes": n_arch,
        "n_observations": len(labels),
        "decode_mean": float(
            row["decode_mean"]
        ),
        "decode_err": float(
            row["decode_err"]
        ),
        "selected_archetypes": str(
            selected
        ),
        "mean_selected_frequency": float(
            row["mean_selected_frequency"]
        ),
        "min_selected_frequency": float(
            row["min_selected_frequency"]
        ),
        "feature_mode": FEATURE_MODE,
        "npz_path": str(
            fit["path"]
        ),
    }

    # --------------------------------------------------------
    # Resume a partial job if one exists
    # --------------------------------------------------------
    if partial_path.exists() and not FORCE_RECOMPUTE:
        job_df = pd.read_csv(partial_path)

        completed_random = set(
            job_df.loc[
                job_df["subset_type"].astype(str) == "random",
                "iter",
            ]
            .astype(int)
            .tolist()
        )

        print(
            f"[{job_i + 1}/{len(jobs)}] RESUME: "
            f"{condition} | K={K} | top-{top_m} | "
            f"{len(completed_random)}/{n_random} random subsets already saved"
        )

    else:
        # ----------------------------------------------------
        # Compute observed consensus subset first and save it
        # immediately.
        # ----------------------------------------------------
        X_top = (
            build_observation_feature_matrix(
                results_subj,
                selected,
            )
        )

        obs_metrics = evaluate_clustering(
            X_top,
            labels,
            seed=RANDOM_SEED,
        )

        job_df = pd.DataFrame([
            {
                **base,
                "subset_type": "crossfit_consensus",
                "iter": -1,
                **obs_metrics,
            }
        ])

        atomic_to_csv(
            job_df,
            partial_path,
        )

        completed_random = set()

        print(
            f"[{job_i + 1}/{len(jobs)}] START: "
            f"{condition} | K={K} | top-{top_m}"
        )

    # --------------------------------------------------------
    # Generate the SAME full set of random subsets every time.
    # This makes resumed runs exactly reproducible.
    # --------------------------------------------------------
    base_seed = (
        RANDOM_SEED
        + 100000 * K
        + 1000 * top_m
        + CONDITIONS.index(condition)
    )

    rng = np.random.default_rng(
        base_seed
    )

    random_subsets = [
        (
            rng.choice(
                np.arange(n_arch),
                size=actual_m,
                replace=False,
            )
            .astype(int)
            .tolist()
        )
        for _ in range(n_random)
    ]

    pending_rows = []

    for ii, rand_sel in enumerate(random_subsets):

        if ii in completed_random:
            continue

        X_rand = (
            build_observation_feature_matrix(
                results_subj,
                rand_sel,
            )
        )

        rand_metrics = evaluate_clustering(
            X_rand,
            labels,
            seed=RANDOM_SEED + ii,
        )

        pending_rows.append({
            **base,
            "subset_type": "random",
            "iter": ii,
            "selected_archetypes": str(
                rand_sel
            ),
            **rand_metrics,
        })

        # ----------------------------------------------------
        # Save progress in small batches.
        # ----------------------------------------------------
        if (
            len(pending_rows) >= CHECKPOINT_EVERY
            or ii == n_random - 1
        ):
            job_df = pd.concat(
                [
                    job_df,
                    pd.DataFrame(pending_rows),
                ],
                ignore_index=True,
            )

            # Remove accidental duplicates if a previous run stopped
            # at an awkward point.
            job_df = (
                job_df
                .drop_duplicates(
                    subset=[
                        "subset_type",
                        "iter",
                    ],
                    keep="last",
                )
                .sort_values(
                    [
                        "subset_type",
                        "iter",
                    ]
                )
                .reset_index(drop=True)
            )

            atomic_to_csv(
                job_df,
                partial_path,
            )

            n_saved = int(
                np.sum(
                    job_df["subset_type"].astype(str) == "random"
                )
            )

            print(
                f"    checkpoint: {condition} | K={K} | top-{top_m} "
                f"| {n_saved}/{n_random} random subsets saved"
            )

            pending_rows = []

    # --------------------------------------------------------
    # Verify job completeness before promoting the checkpoint.
    # --------------------------------------------------------
    n_saved = int(
        np.sum(
            job_df["subset_type"].astype(str) == "random"
        )
    )

    has_observed = bool(
        np.any(
            job_df["subset_type"].astype(str)
            == "crossfit_consensus"
        )
    )

    if n_saved == n_random and has_observed:

        atomic_to_csv(
            job_df,
            complete_path,
        )

        if partial_path.exists():
            partial_path.unlink()

        print(
            f"[{job_i + 1}/{len(jobs)}] COMPLETE: "
            f"{condition} | K={K} | top-{top_m}"
        )

    else:
        print(
            f"[{job_i + 1}/{len(jobs)}] INCOMPLETE but safely checkpointed: "
            f"{condition} | K={K} | top-{top_m} | "
            f"{n_saved}/{n_random}"
        )


# ============================================================
# REBUILD THE MASTER LONG TABLE FROM COMPLETED JOB FILES
# ============================================================

complete_files = sorted(
    CHECKPOINT_DIR.glob("*.csv")
)

# Exclude any accidental "*.partial.csv" matches.
complete_files = [
    p for p in complete_files
    if not p.name.endswith(".partial.csv")
]

if len(complete_files) == 0:
    clustering_long_df = pd.DataFrame()
    print("No completed jobs yet.")

else:
    clustering_long_df = pd.concat(
        [
            pd.read_csv(p)
            for p in complete_files
        ],
        ignore_index=True,
    )

    atomic_to_csv(
        clustering_long_df,
        LONG_CSV,
    )

    print(
        f"\nMaster long table rebuilt from {len(complete_files)} "
        f"completed jobs: {LONG_CSV}"
    )

    print(
        "Master long rows:",
        len(clustering_long_df),
    )


skipped_df = pd.DataFrame(
    skipped
)

if len(skipped_df):
    display(skipped_df)

In [ ]:
# ============================================================
# SUMMARIZE SELECTED VS RANDOM
#
# Restart-safe: reload completed checkpoint files directly.
# ============================================================

complete_files = sorted(
    p for p in CHECKPOINT_DIR.glob("*.csv")
    if not p.name.endswith(".partial.csv")
)

if len(complete_files) == 0:
    raise RuntimeError(
        "No completed clustering jobs are available yet. "
        "Run/resume the checkpointed clustering cell first."
    )

clustering_long_df = pd.concat(
    [pd.read_csv(p) for p in complete_files],
    ignore_index=True,
)

atomic_to_csv(
    clustering_long_df,
    LONG_CSV,
)

print(
    f"Loaded {len(complete_files)} completed checkpoint files "
    f"({len(clustering_long_df)} rows)."
)

METRICS = [
    "purity",
    "balance",
    "purity_x_balance",
    "ari",
    "nmi",
    "nearest_centroid_accuracy",
    "silhouette_true_labels",
    "silhouette_cluster_labels",
]

group_cols = [
    "analysis_type",
    "fit_scope",
    "K",
    "component_ratio",
    "condition",
    "top_m",
    "actual_m",
    "decode_mean",
    "decode_err",
    "mean_selected_frequency",
    "min_selected_frequency",
    "feature_mode",
]

summary_rows = []

for keys, sub in clustering_long_df.groupby(
    group_cols,
    dropna=False,
):
    key_dict = dict(
        zip(
            group_cols,
            keys,
        )
    )

    obs = sub[
        sub["subset_type"]
        == "crossfit_consensus"
    ]

    rnd = sub[
        sub["subset_type"]
        == "random"
    ]

    if len(obs) == 0 or len(rnd) == 0:
        continue

    obs_row = obs.iloc[0]

    out = dict(
        key_dict
    )

    for metric in METRICS:
        obs_val = float(
            obs_row[metric]
        )

        rand_vals = rnd[
            metric
        ].to_numpy(
            dtype=float
        )

        rand_vals = rand_vals[
            np.isfinite(
                rand_vals
            )
        ]

        out[
            f"{metric}_observed"
        ] = obs_val

        if (
            len(rand_vals) == 0
            or not np.isfinite(
                obs_val
            )
        ):
            out[
                f"{metric}_random_mean"
            ] = np.nan

            out[
                f"{metric}_random_std"
            ] = np.nan

            out[
                f"{metric}_z"
            ] = np.nan

            out[
                f"{metric}_p_high"
            ] = np.nan

            out[
                f"{metric}_observed_minus_random"
            ] = np.nan

            continue

        rand_mean = float(
            np.mean(
                rand_vals
            )
        )

        rand_std = float(
            np.std(
                rand_vals
            )
        )

        out[
            f"{metric}_random_mean"
        ] = rand_mean

        out[
            f"{metric}_random_std"
        ] = rand_std

        out[
            f"{metric}_z"
        ] = float(
            (
                obs_val
                - rand_mean
            )
            / (
                rand_std
                + 1e-12
            )
        )

        out[
            f"{metric}_p_high"
        ] = float(
            (
                np.sum(
                    rand_vals >= obs_val
                )
                + 1
            )
            / (
                len(
                    rand_vals
                )
                + 1
            )
        )

        out[
            f"{metric}_observed_minus_random"
        ] = float(
            obs_val
            - rand_mean
        )

    summary_rows.append(
        out
    )


clustering_summary_df = pd.DataFrame(
    summary_rows
)

clustering_summary_df.to_csv(
    SUMMARY_CSV,
    index=False,
)

print("Summary rows:", len(clustering_summary_df))
print("Saved:", SUMMARY_CSV)

display(
    clustering_summary_df
    .sort_values(
        "nmi_observed_minus_random",
        ascending=False,
    )
    .head(20)
)

In [ ]:
# ============================================================
# RECOMPUTE THE ORIGINAL JOINT SCORE USING HELD-OUT DECODING
# ============================================================

def zscore_safe(x):
    x = np.asarray(
        x,
        dtype=float,
    )

    mu = np.nanmean(
        x
    )

    sd = np.nanstd(
        x
    )

    if (
        not np.isfinite(sd)
        or sd < 1e-12
    ):
        return np.zeros_like(
            x,
            dtype=float,
        )

    return (
        x - mu
    ) / sd


joint_df = (
    clustering_summary_df
    .copy()
)

joint_df["decode_z"] = np.nan
joint_df["cluster_effect_z"] = np.nan

# Match the old notebook's primary normalization:
# within ranking condition and clustering metric.
for condition, idx in (
    joint_df
    .groupby(
        "condition"
    )
    .groups
    .items()
):
    idx = list(
        idx
    )

    joint_df.loc[
        idx,
        "decode_z",
    ] = zscore_safe(
        joint_df.loc[
            idx,
            "decode_mean",
        ]
    )

    joint_df.loc[
        idx,
        "cluster_effect_z",
    ] = zscore_safe(
        joint_df.loc[
            idx,
            f"{PRIMARY_CLUSTER_METRIC}_observed_minus_random",
        ]
    )


joint_df["joint_zsum"] = (
    joint_df["decode_z"]
    + joint_df["cluster_effect_z"]
)

display(
    joint_df
    .sort_values(
        "joint_zsum",
        ascending=False,
    )
    .head(20)[
        [
            "condition",
            "K",
            "top_m",
            "decode_mean",
            "decode_z",
            f"{PRIMARY_CLUSTER_METRIC}_observed_minus_random",
            "cluster_effect_z",
            "joint_zsum",
            "mean_selected_frequency",
        ]
    ]
)

In [ ]:
# ============================================================
# PAPER-READY TWO-PANEL FIGURE
#
# Intact is shown here to match the simplified paper narrative.
# Change FIGURE_CONDITION if desired.
# ============================================================

FIGURE_CONDITION = "intact"

plot_df = (
    joint_df[
        joint_df["condition"]
        == FIGURE_CONDITION
    ]
    .copy()
)

if len(plot_df) == 0:
    raise ValueError(
        f"No rows for {FIGURE_CONDITION}"
    )

all_k = sorted(
    plot_df["K"]
    .astype(int)
    .unique()
)

k_to_rank = {
    k: i
    for i, k
    in enumerate(all_k)
}

plot_df["K_rank"] = (
    plot_df["K"]
    .map(k_to_rank)
)

marker_map = {
    3: "o",
    5: "s",
    10: "^",
}

# Fallback for any other m values.
fallback_markers = [
    "D",
    "P",
    "X",
    "v",
    "*",
]

for m in sorted(
    plot_df["top_m"]
    .unique()
):
    if m not in marker_map:
        marker_map[m] = (
            fallback_markers[
                len(marker_map)
                % len(fallback_markers)
            ]
        )


fig = plt.figure(
    figsize=(9.0, 10.0)
)

gs = fig.add_gridspec(
    2,
    1,
    height_ratios=[
        1.18,
        0.88,
    ],
    hspace=0.30,
)

axA = fig.add_subplot(
    gs[0, 0]
)

axB = fig.add_subplot(
    gs[1, 0]
)


# ------------------------------------------------------------
# A — held-out decoding vs clustering effect
# ------------------------------------------------------------

cmap = plt.cm.viridis

norm = plt.Normalize(
    vmin=0,
    vmax=max(
        1,
        len(all_k) - 1,
    ),
)

for top_m in sorted(
    plot_df["top_m"]
    .unique()
):
    cur = plot_df[
        plot_df["top_m"]
        == top_m
    ]

    axA.scatter(
        cur["decode_z"],
        cur["cluster_effect_z"],
        c=cur["K_rank"],
        cmap=cmap,
        norm=norm,
        s=92,
        marker=marker_map[
            int(top_m)
        ],
        alpha=0.88,
        edgecolor="0.30",
        linewidth=0.45,
        label=f"top-{int(top_m)}",
        zorder=3,
    )


axA.axhline(
    0,
    color="0.60",
    linestyle="--",
    linewidth=1.0,
)

axA.axvline(
    0,
    color="0.60",
    linestyle="--",
    linewidth=1.0,
)


top_cases = (
    plot_df
    .sort_values(
        "joint_zsum",
        ascending=False,
    )
    .head(6)
)

for _, r in top_cases.iterrows():
    axA.annotate(
        f"K={int(r['K'])}",
        (
            r["decode_z"],
            r["cluster_effect_z"],
        ),
        xytext=(4, 5),
        textcoords="offset points",
        fontsize=8.5,
    )


axA.set_xlabel(
    "Held-out decoding z-score",
    fontsize=12,
)

axA.set_ylabel(
    "Clustering-effect z-score",
    fontsize=12,
)

axA.set_title(
    "Cross-fitted decoding subsets and condition-related clustering",
    fontsize=13,
    fontweight="bold",
)

axA.legend(
    title="Subset size",
    frameon=False,
    loc="lower right",
    ncol=1,
)


sm = plt.cm.ScalarMappable(
    norm=norm,
    cmap=cmap,
)

cbar = fig.colorbar(
    sm,
    ax=axA,
    fraction=0.035,
    pad=0.025,
)

tick_step = max(
    1,
    len(all_k) // 10,
)

tick_positions = np.arange(
    0,
    len(all_k),
    tick_step,
)

if (
    len(tick_positions) == 0
    or tick_positions[-1]
    != len(all_k) - 1
):
    tick_positions = np.append(
        tick_positions,
        len(all_k) - 1,
    )

cbar.set_ticks(
    tick_positions
)

cbar.set_ticklabels([
    str(
        all_k[int(i)]
    )
    for i in tick_positions
])

cbar.set_label(
    "Model order K",
    fontsize=10.5,
)


# ------------------------------------------------------------
# B — joint held-out decoding + clustering score
# ------------------------------------------------------------

heat = (
    plot_df
    .pivot_table(
        index="top_m",
        columns="K",
        values="joint_zsum",
        aggfunc="mean",
    )
    .reindex(
        index=sorted(
            plot_df["top_m"]
            .unique()
        )
    )
    .reindex(
        columns=all_k
    )
)

arr = heat.to_numpy(
    dtype=float
)

vmax = np.nanmax(
    np.abs(
        arr
    )
)

if (
    not np.isfinite(vmax)
    or vmax == 0
):
    vmax = 1.0


im = axB.imshow(
    arr,
    aspect="auto",
    cmap="coolwarm",
    vmin=-vmax,
    vmax=vmax,
    interpolation="nearest",
)

axB.set_xticks(
    np.arange(
        len(all_k)
    )
)

axB.set_xticklabels(
    [
        str(k)
        for k in all_k
    ],
    rotation=55,
    ha="right",
    fontsize=8,
)

topm_sorted = sorted(
    plot_df["top_m"]
    .unique()
)

axB.set_yticks(
    np.arange(
        len(topm_sorted)
    )
)

axB.set_yticklabels(
    [
        str(int(m))
        for m in topm_sorted
    ]
)

axB.set_xlabel(
    "Model order K",
    fontsize=12,
)

axB.set_ylabel(
    "top-m",
    fontsize=12,
)

axB.set_title(
    "Joint held-out decoding + clustering score",
    fontsize=13,
    fontweight="bold",
)

cb = fig.colorbar(
    im,
    ax=axB,
    fraction=0.035,
    pad=0.025,
)

cb.set_label(
    "Joint z-score sum",
    fontsize=10.5,
)


for ax, letter in [
    (axA, "A"),
    (axB, "B"),
]:
    ax.text(
        -0.07,
        1.04,
        letter,
        transform=ax.transAxes,
        fontsize=17,
        fontweight="bold",
        va="top",
    )


fig.suptitle(
    "Held-out-selected spatial archetype subsets and condition structure",
    fontsize=15,
    fontweight="bold",
    y=0.985,
)

fig.subplots_adjust(
    left=0.11,
    right=0.93,
    top=0.92,
    bottom=0.08,
)

fig.savefig(
    FIG_PNG,
    dpi=DPI,
    bbox_inches="tight",
)

fig.savefig(
    FIG_PDF,
    bbox_inches="tight",
)

plt.show()

print("Saved:", FIG_PNG)
print("Saved:", FIG_PDF)

## How I would use this in the revised paper

This should replace the old same-data decoding–clustering figure if the pattern
survives.

The strongest defensible wording would be along the lines of:

> Archetype subsets identified reproducibly across participant-level
> cross-fitting also exhibited stronger condition-related organization than
> random subsets of the same size and model order.

That wording is preferable to saying the clustering analysis itself is
held-out, because the cross-fitting applies to **subset selection and decoding**;
the condition-clustering validation retains the original all-participant
geometry analysis.